# Pipeline de Água — Projeto PI
Estágio, FEUP

## O que faz
Pipeline responsável por download, parsing, limpeza, deteção de anomalias, persistência em base de dados e envio de relatório por email de dados de contadores de água.

Funciona em vários ambientes: **Google Colab**, **Render** e **Flask local**.

**Para qualquer erro/sugestão deve entrar em contacto com um dos membros do Grupo PI.**

---

## Secrets necessários no Colab

### Obrigatório (uma destas opções)
| Secret | Exemplo | Descrição |
|---|---|---|
| `GITHUB_TOKEN` | `ghp_...` | Token GitHub direto (recomendado) |
| `<USER>_github_json` | `{"key": "ghp_..."}` | Formato legado por prefixo |

### Recomendado
| Secret | Exemplo | Descrição |
|---|---|---|
| `USER` | `TO` ou `PI` | Prefixo do grupo. Só é necessário para `<USER>_github_json` |

### Fonte de dados (escolher uma)
| Secret | Exemplo | Descrição |
|---|---|---|
| `REPO_OWNER` | `pedroccpimenta` | Owner do repo de dados |
| `REPO_NAME` | `datafiles` | Nome do repo de dados |
| `REPO_FOLDER` | `aqualog` | Pasta dentro do repo com os JSONs |
| `REPO_BRANCH` | `master` | Branch do repo de dados |

### Email (opcional)
| Secret | Exemplo | Descrição |
|---|---|---|
| `EMAIL_ENABLED` | `true` | Ativar envio de email |
| `EMAIL_TO` | `user@example.com` | Destinatário(s), separados por vírgula |
| `BREVO_API_KEY` | `xkeysib-...` | Chave API do Brevo (recomendado) |
| `BREVO_SENDER_NAME` | `PI Water` | Nome do remetente |
| `BREVO_SENDER_EMAIL` | `noreply@example.com` | Email do remetente |

### Bases de dados (opcional — ativar as que forem usar)
| Secret | Exemplo | Descrição |
|---|---|---|
| `MONGO_URI` | `mongodb+srv://...` | URI do MongoDB Atlas |
| `TIDB_HOST` | `gateway01.eu-central-1...` | Host do TiDB Cloud |
| `TIDB_USER` | `username.root` | Utilizador TiDB |
| `TIDB_PASSWORD` | `password` | Password TiDB |
| `TIDB_DATABASE` | `pi_water` | Base de dados TiDB |
| `CRATEDB_HOST` | `https://...cratedb.net:4200` | Host do CrateDB |
| `CRATEDB_USER` | `admin` | Utilizador CrateDB |
| `CRATEDB_PASSWORD` | `password` | Password CrateDB |

### Outras opções
| Secret | Default | Descrição |
|---|---|---|
| `SKIP_DB_WRITES` | `false` | Ignorar escrita nas BDs (útil para testes) |
| `DAYS_BACK` | `0` | Nº de dias para trás na descoberta de ficheiros |
| `ANOMALY_LOOKBACK_DAYS` | `2` | Dias de histórico para deteção de anomalias |
| `VERBOSE` | `true` | Logging detalhado |

### Compatibilidade com formato legado (grupo)
Também é suportado o formato do teu colega:
- `TO_github_json`
- `TO_tidb_json`
- `TO_crate_json`
- `TO_mongodb_json`
- `configGMail_TO_json`
- `EMAIL_SEND`, `EMAIL_ADDRESSES`

---

Grupo PI, 2026


In [ ]:
from google.colab import userdata
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


def get_secret(name, default=''):
    try:
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return default


def extract_github_token(raw):
    text = str(raw or '').strip()
    if not text:
        return ''

    # Aceita token direto (ghp_...) e tambem JSON ({"key":"ghp_..."}).
    try:
        payload = json.loads(text)
        if isinstance(payload, dict):
            token_value = payload.get('key') or payload.get('token') or payload.get('value') or ''
            return str(token_value).strip()
        if isinstance(payload, str):
            return payload.strip()
    except Exception:
        pass

    return text


def run_cmd(cmd, cwd='/content'):
    return subprocess.run(cmd, check=False, capture_output=True, text=True, cwd=cwd)


def clone_repo(repo_url_no_auth, token, dest_path):
    # 1) Tenta clone autenticado via header (nao expoe token na URL/logs)
    auth_cmd = [
        'git',
        '-c',
        f'http.extraheader=Authorization: Bearer {token}',
        'clone',
        repo_url_no_auth,
        str(dest_path),
    ]
    auth_result = run_cmd(auth_cmd)
    if auth_result.returncode == 0:
        return

    # 2) Fallback sem token para perceber se o repo e publico
    public_cmd = ['git', 'clone', repo_url_no_auth, str(dest_path)]
    public_result = run_cmd(public_cmd)
    if public_result.returncode == 0:
        return

    auth_err = (auth_result.stderr or auth_result.stdout or '').strip()
    public_err = (public_result.stderr or public_result.stdout or '').strip()

    raise RuntimeError(
        'Falha no git clone.\n'
        f'Com token: {auth_err[:800]}\n'
        f'Sem token: {public_err[:800]}\n\n'
        'Checklist:\n'
        '- Confirma que GITHUB_TOKEN tem permissao de leitura ao repo (repo/contents:read).\n'
        '- Se for Fine-grained PAT, adiciona acesso ao repo HenriquePerry/water_pipeline.\n'
        '- Se o token foi exposto/revogado, gera um novo e atualiza o secret no Colab.'
    )


# 1) Prioridade: secret direto GITHUB_TOKEN
# 2) Fallback: <USER>_github_json (legado)
user = str(get_secret('USER', '')).strip() or 'TO'
token = extract_github_token(get_secret('GITHUB_TOKEN', ''))

if not token:
    token_json = get_secret(f"{user}_github_json", '')
    if not token_json and user != 'PI':
        user = 'PI'
        token_json = get_secret(f"{user}_github_json", '')

    if not token_json:
        raise RuntimeError(
            'Nao encontrei GITHUB_TOKEN nem <USER>_github_json. '
            'Define GITHUB_TOKEN (recomendado) ou TO_github_json/PI_github_json.'
        )

    token = extract_github_token(token_json)
    if not token:
        raise RuntimeError("Secret github_json invalido: esperado JSON com campo 'key' (ou 'token').")

repo_name = 'water_pipeline'
repo_url_no_auth = 'https://github.com/HenriquePerry/water_pipeline.git'
repo_path = Path('/content') / repo_name

# Garante que nunca apagamos o repo estando dentro dele
os.chdir('/content')

if repo_path.exists():
    shutil.rmtree(repo_path)

clone_repo(repo_url_no_auth, token, repo_path)

if not repo_path.exists():
    raise RuntimeError('Clone do repositorio falhou: pasta /content/water_pipeline nao foi criada.')

app_file = repo_path / 'app.py'
if not app_file.exists():
    candidates = ', '.join(sorted(p.name for p in repo_path.iterdir())[:30])
    raise RuntimeError(
        'Clone concluido, mas app.py nao foi encontrado em /content/water_pipeline. '
        f'Conteudo atual: {candidates}'
    )

os.chdir(repo_path)
if str(repo_path) in sys.path:
    sys.path.remove(str(repo_path))
sys.path.insert(0, str(repo_path))
importlib.invalidate_caches()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

# Garante que o runtime encontra o token no formato direto
os.environ['GITHUB_TOKEN'] = token
os.environ['USER'] = user
os.environ['GITHUB_SECRET_NAME'] = f"{user}_github_json"

# Injeta secrets de email no env antes do import do app/config.
for secret_name in [
    'EMAIL_ENABLED',
    'EMAIL_BACKEND',
    'EMAIL_TO',
    'EMAIL_FROM',
    'EMAIL_USERNAME',
    'EMAIL_PASSWORD',
    'EMAIL_SMTP_HOST',
    'EMAIL_SMTP_PORT',
    'BREVO_API_KEY',
    'BREVO_SENDER_EMAIL',
    'BREVO_SENDER_NAME',
    'BREVO_USER',
    'BREVO_PASSWORD',
    'BREVO_FROM',
    'EMAIL_SEND',
    'EMAIL_ADDRESSES',
]:
    value = str(get_secret(secret_name, '')).strip()
    if value:
        os.environ[secret_name] = value

# Compatibilidade com naming legado
if not os.environ.get('EMAIL_ENABLED') and os.environ.get('EMAIL_SEND'):
    os.environ['EMAIL_ENABLED'] = os.environ['EMAIL_SEND']
if not os.environ.get('EMAIL_TO') and os.environ.get('EMAIL_ADDRESSES'):
    os.environ['EMAIL_TO'] = os.environ['EMAIL_ADDRESSES']
if not os.environ.get('EMAIL_FROM') and os.environ.get('BREVO_FROM'):
    os.environ['EMAIL_FROM'] = os.environ['BREVO_FROM']
if not os.environ.get('EMAIL_FROM') and os.environ.get('BREVO_SENDER_EMAIL'):
    os.environ['EMAIL_FROM'] = os.environ['BREVO_SENDER_EMAIL']
if not os.environ.get('EMAIL_BACKEND'):
    if os.environ.get('BREVO_API_KEY') and os.environ.get('BREVO_SENDER_EMAIL'):
        os.environ['EMAIL_BACKEND'] = 'brevo'
    else:
        os.environ['EMAIL_BACKEND'] = 'smtp'

# Defaults para SMTP Brevo quando credenciais SMTP existem
if not os.environ.get('EMAIL_SMTP_HOST') and (
    os.environ.get('BREVO_USER') and os.environ.get('BREVO_PASSWORD')
):
    os.environ['EMAIL_SMTP_HOST'] = 'smtp-relay.brevo.com'
if not os.environ.get('EMAIL_SMTP_PORT') and (
    os.environ.get('BREVO_USER') and os.environ.get('BREVO_PASSWORD')
):
    os.environ['EMAIL_SMTP_PORT'] = '587'

print('EMAIL PRECHECK (before import app):', {
    'EMAIL_ENABLED': os.environ.get('EMAIL_ENABLED', ''),
    'EMAIL_BACKEND': os.environ.get('EMAIL_BACKEND', ''),
    'EMAIL_TO_count': len([x.strip() for x in os.environ.get('EMAIL_TO', '').split(',') if x.strip()]),
    'has_EMAIL_FROM': bool(os.environ.get('EMAIL_FROM')),
    'has_EMAIL_USERNAME': bool(os.environ.get('EMAIL_USERNAME')),
    'has_EMAIL_PASSWORD': bool(os.environ.get('EMAIL_PASSWORD')),
    'has_BREVO_API_KEY': bool(os.environ.get('BREVO_API_KEY')),
    'has_BREVO_SENDER_EMAIL': bool(os.environ.get('BREVO_SENDER_EMAIL')),
})

# Muito importante em Colab: evita usar CONFIG em cache de imports anteriores.
for module_name in ['app', 'pipeline', 'notifier', 'config', 'env_utils']:
    if module_name in sys.modules:
        del sys.modules[module_name]
importlib.invalidate_caches()

app = importlib.import_module('app')

# Mostra checks exatamente como a app ve no runtime.
runtime_checks = app._email_readiness() if hasattr(app, '_email_readiness') else {}
print('EMAIL READINESS (app):', json.dumps(runtime_checks, indent=2, ensure_ascii=False, default=str))

if hasattr(app, 'CONFIG'):
    print('CONFIG EMAIL SNAPSHOT:', {
        'email_enabled': app.CONFIG.get('email_enabled'),
        'email_backend': app.CONFIG.get('email_backend'),
        'email_to': bool(app.CONFIG.get('email_to')),
        'brevo_api_key': bool(app.CONFIG.get('brevo_api_key')),
        'brevo_sender_email': bool(app.CONFIG.get('brevo_sender_email')),
    })

result = app.run_pipeline_now(runtime_label='google-colab', send_email=True)
print(json.dumps(result, indent=2, ensure_ascii=False, default=str))

email_dispatch = result.get('email_dispatch') or {}
print('\nEMAIL_DISPATCH:', json.dumps(email_dispatch, indent=2, ensure_ascii=False, default=str))